In [1]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: c:\Users\123\Desktop\start_vector
✅ Sys path: ['c:\\Users\\123\\Desktop\\start_vector', 'C:\\Python314\\python314.zip', 'C:\\Python314\\DLLs']...


In [3]:
from src_oop.core.database import Database
from src_oop.jobs.unit.queries import query_adv_spend
from src_oop.jobs.unit.config import unit_gs
from src_oop.jobs.fin_reports_analyze.repository import FinReportsAnalyze
from src_oop.core.my_gspread import GoogleTabs

import pandas as pd

In [ ]:
# Инициализируем базу данных и получаем данные о рекламных расходах
database = Database()
# Получаем данные о рекламных расходах по статьям за вчерашний день
adv_spend = database.read_sql_to_dataframe(query_adv_spend)
# Задаем параметры для работы с Google Sheets
table_title = unit_gs.get("title")
sheet_title = unit_gs.get("unit_sheet")
# Инициализируем работу с Google Sheets
google_tabs = GoogleTabs(table_title, sheet_title)
# Получаем данные из Google Sheets и преобразуем их в DataFrame
sheet_data = google_tabs.sheet_title.get_all_values()
headers = sheet_data[0]
rows = sheet_data[1:]
df = pd.DataFrame(rows, columns=headers)
# Приводим столбец 'Артикул' к типу int для корректного сравнения
df['Артикул'] = df['Артикул'].astype(int)
# Создаем датафрейм с нужными столбцами для дальнейшей работы
df_short = df[['Артикул', 'Реклама']]

# 1. Получаем список всех уникальных артикулов, по которым БЫЛИ затраты
articles_with_spend = set(adv_spend['article_id'].astype(int))

# 2. Создаем функцию для проверки
def check_adv(article):
    # Приводим к строке для надежности сравнения
    if int(article) in articles_with_spend:
        return "реклама"
    else:
        return ""

# 3. Применяем функцию к колонке 'Артикул' в df_short
df_short['Реклама'] = df_short['Артикул'].apply(check_adv)

# Динамически находим индекс колонки "Артикул", чтобы не ошибиться
if 'Артикул' not in headers:
    print("Ошибка: В таблице нет колонки 'Артикул'!")
else:
    art_idx = headers.index('Артикул')

    # --- 3. Сопоставление ---
    # Важно: мы создаем список значений в том же порядке, в котором идут строки в таблице
    results_list = []
    for row in rows:
        # Проверяем артикул из текущей строки таблицы
        current_art = int(row[art_idx])
        
        if current_art in articles_with_spend:
            results_list.append("реклама")
        else:
            results_list.append("")

    # --- 4. Запись ---
    # Теперь нам всё равно, где находится колонка "Реклама", метод сам её найдет
    google_tabs.update_column_by_name("Реклама", results_list)


✅ Успешное подключение к UNIT 2.0 (tested) -> MAIN (tested)


C:\Users\123\AppData\Local\Temp\ipykernel_19784\4207690136.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_short['Реклама'] = df_short['Артикул'].apply(check_adv)


✅ Данные успешно записаны в колонку 'Реклама' (диапазон E2:E7285)


In [18]:
query = """
    SELECT c.target_vendor_code,
       c.target_account,
       c.local_vendor_code,
       a.nm_id,
       cd.subject_name,
       cd.wb_name,
       cp.purchase_price,
       DATE(c.updated_at) as date
FROM card_clone_job c
LEFT JOIN article a
    ON c.target_vendor_code = a.vendor_code
    AND c.target_account = a.account
LEFT JOIN (
    SELECT AVG(cp.purchase_price) AS purchase_price,
           cp.local_vendor_code,
           cp.date
        FROM cost_price cp
        WHERE cp.date >= '2026-04-14'
        GROUP BY cp.local_vendor_code, cp.date
    ) cp
    ON c.local_vendor_code = cp.local_vendor_code
    AND DATE(c.updated_at) = cp.date
LEFT JOIN card_data cd
    ON a.nm_id = cd.article_id
WHERE DATE(c.updated_at) >= '2026-04-14'
AND a.nm_id IS NOT NULL;
       """
analyze = FinReportsAnalyze()
df = analyze.get_df_from_db(query)

In [20]:
df['target_account'] = df['target_account'].str.capitalize()
df.head()

,target_vendor_code,target_account,local_vendor_code,nm_id,subject_name,wb_name,purchase_price,date
0,wild115,Вектор8830,wild115,963998938,Бритвы электрические,Электробритва DEM-TX200,NaN,2026-04-14
1,wild206,Вектор8830,wild206,963998457,Ирригаторы,Ирригатор портативный для зубов W3Pro,3442.0,2026-04-14
2,wild831,Вектор8830,wild831,963998643,Чайники электрические,"Чайник электрический DЕЕMA DEM-SH50W 1,7 л",3108.0,2026-04-14
3,wild1496,Вектор8830,wild1496,963998687,Велотренажеры,Велотренажер мини для рук и ног,1320.0,2026-04-14
4,wild1152,Вектор8830,wild1152,963998843,Масленки,Масленка стальная с прозрачной крышкой,315.0,2026-04-14


In [21]:
filtered_df = df[~(df['local_vendor_code'] != df['target_vendor_code']) & (df['local_vendor_code'].str.contains('wild', na=False))]
filtered_df.shape

(1799, 8)

In [22]:
path = "C:\\Users\\123\\Downloads\\filtered_data.csv"
filtered_df.to_csv(path, index=False)

In [13]:
filtered_df.tail()

,target_vendor_code,target_account,local_vendor_code,nm_id,purchase_price,date
2200,wild1169,СТАРТ7528,wild1169,NaN,970.0,2026-04-15
2201,wild831,СТАРТ7528,wild831,NaN,3108.0,2026-04-14
2202,wild447,СТАРТ7528,wild447,NaN,564.0,2026-04-14
2203,wild736,ВЕКТОР8830,wild736,NaN,NaN,2026-04-14
2204,wild1232,СТАРТ7528,wild1232,NaN,0.0,2026-04-15
